# Module 5 — Biological Network Graph

This notebook builds an interactive network graph visualising the relationships between:

**Host Plant → Endophyte → Compound → Disease/Bioactivity**

Built using **NetworkX** (network analysis) and **Plotly** (interactive visualisation).

**This notebook is independent — no other notebook needs to run first.**

In [ ]:
# ── CELL 1: Install libraries ──
!pip install networkx plotly --quiet
print("✓ Libraries installed")

In [ ]:
# ── CELL 2: Import libraries ──
import networkx as nx
import plotly.graph_objects as go
import pandas as pd
import os

os.makedirs("figures", exist_ok=True)
print("✓ Ready")

In [ ]:
# ── CELL 3: Define network data ──
# Each entry = one relationship chain:
# Host Plant → Endophyte → Compound → Bioactivity
# Source: Banerjee (2019), NTCC Review, Amity University Kolkata

edges_data = [
    # (Host Plant, Endophyte, Compound, Bioactivity)
    ("Taxus brevifolia",         "Taxomyces andreanae",   "Taxol",           "Anticancer"),
    ("Nothapodytes foetida",     "Fusarium solani",        "Camptothecin",    "Anticancer"),
    ("Podophyllum hexandrum",    "Trametes hirsuta",       "Podophyllotoxin", "Anticancer"),
    ("Mimusops elengi",          "Unidentified fungi",     "Ergoflavin",      "Anticancer"),
    ("Torreya taxifolia",        "Pestalotiopsis microspora", "Torreyanic acid", "Anticancer"),
    ("Terminalia morobensis",    "Pestalotiopsis microspora", "Pestacin",     "Antioxidant"),
    ("Terminalia morobensis",    "Pestalotiopsis microspora", "Isopestacin",  "Antioxidant"),
    ("Cinchona pubescens",       "Xylaria sp.",            "Griseofulvin",    "Antimicrobial"),
    ("Monstera sp.",             "Streptomyces sp.",       "Coronamycin",     "Antimicrobial"),
    ("Hypericum perforatum",     "Diaporthe helianthi",   "Hypericin",       "Antimicrobial"),
    ("Hypericum perforatum",     "Diaporthe helianthi",   "Emodin",          "Antimicrobial"),
    ("Grevillea pteridifolia",   "Streptomyces sp.",       "Kakadumycin A",   "Antimicrobial"),
    ("Quercus variabilis",       "Cladosporium sp.",       "Brefeldin A",     "Antimicrobial"),
    ("Various",                  "Cytonaema sp.",          "Helvolic acid",   "Antimicrobial"),
    ("Huperzia serrata",         "Acremonium sp.",         "Huperzine A",     "Neuroprotective"),
    ("Tripterygium wilfordii",   "Fusarium subglutinans", "Subglutinol A",   "Immunosuppressive"),
]

print(f"✓ Network data defined: {len(edges_data)} relationship chains")

In [ ]:
# ── CELL 4: Build NetworkX graph ──
G = nx.DiGraph()  # directed graph (arrows show direction)

# Define node types for colour coding
node_types = {}

for host, endophyte, compound, bioactivity in edges_data:
    # Add nodes
    G.add_node(host,        node_type="Host Plant")
    G.add_node(endophyte,   node_type="Endophyte")
    G.add_node(compound,    node_type="Compound")
    G.add_node(bioactivity, node_type="Bioactivity")

    # Add edges (directed)
    G.add_edge(host,       endophyte,  label="harbours")
    G.add_edge(endophyte,  compound,   label="produces")
    G.add_edge(compound,   bioactivity, label="shows")

    node_types[host]        = "Host Plant"
    node_types[endophyte]   = "Endophyte"
    node_types[compound]    = "Compound"
    node_types[bioactivity] = "Bioactivity"

print(f"✓ Graph built")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")

In [ ]:
# ── CELL 5: Layout and colour scheme ──
import math

# Use spring layout for organic feel
pos = nx.spring_layout(G, seed=42, k=2.5)

# Colour each node type differently
color_map = {
    "Host Plant":       "#2E7D32",   # dark green
    "Endophyte":        "#1565C0",   # dark blue
    "Compound":         "#F57F17",   # amber
    "Bioactivity":      "#C62828",   # dark red
}

size_map = {
    "Host Plant":   20,
    "Endophyte":    18,
    "Compound":     22,
    "Bioactivity":  25,
}

node_colors = [color_map[node_types[n]] for n in G.nodes()]
node_sizes  = [size_map[node_types[n]]  for n in G.nodes()]

print("✓ Layout computed")

In [ ]:
# ── CELL 6: Build interactive Plotly figure ──

# Build edge traces
edge_traces = []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_traces.append(
        go.Scatter(
            x=[x0, x1, None], y=[y0, y1, None],
            mode="lines",
            line=dict(width=1.2, color="#BDBDBD"),
            hoverinfo="none",
            showlegend=False
        )
    )

# Build one node trace per node type (for legend)
node_traces = []
for ntype, color in color_map.items():
    nodes_of_type = [n for n in G.nodes() if node_types[n] == ntype]
    if not nodes_of_type:
        continue
    x_vals = [pos[n][0] for n in nodes_of_type]
    y_vals = [pos[n][1] for n in nodes_of_type]
    size   = size_map[ntype]

    node_traces.append(
        go.Scatter(
            x=x_vals, y=y_vals,
            mode="markers+text",
            name=ntype,
            text=nodes_of_type,
            textposition="top center",
            textfont=dict(size=9, color="#333"),
            hovertext=[
                f"<b>{n}</b><br>Type: {ntype}<br>Connections: {G.degree(n)}"
                for n in nodes_of_type
            ],
            hoverinfo="text",
            marker=dict(
                size=size,
                color=color,
                line=dict(width=2, color="white")
            )
        )
    )

# Assemble figure
fig = go.Figure(
    data=edge_traces + node_traces,
    layout=go.Layout(
        title=dict(
            text="<b>Endophyte Bioactive Compound Network</b><br>"
                 "<sup>Host Plant → Endophyte → Compound → Bioactivity</sup>",
            x=0.5, font=dict(size=16)
        ),
        showlegend=True,
        legend=dict(title="Node Type", font=dict(size=11)),
        hovermode="closest",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        plot_bgcolor="white",
        paper_bgcolor="white",
        height=750,
        margin=dict(t=80, b=20, l=20, r=20)
    )
)

fig.show()
print("✓ Interactive network graph displayed!")
print("  Hover over any node to see details.")
print("  You can zoom, pan, and click the legend to hide/show node types.")

In [ ]:
# ── CELL 7: Save as HTML (interactive) and PNG (static) ──

# Save interactive HTML — this is the best version to show recruiters!
fig.write_html("figures/network_graph.html")
print("✓ Interactive HTML saved to figures/network_graph.html")

# Save static PNG for README
try:
    fig.write_image("figures/network_graph.png", scale=2)
    print("✓ Static PNG saved to figures/network_graph.png")
except Exception:
    !pip install kaleido --quiet
    fig.write_image("figures/network_graph.png", scale=2)
    print("✓ Static PNG saved to figures/network_graph.png")

In [ ]:
# ── CELL 8: Network statistics ──
print("=== Network Statistics ===")
print(f"Total nodes: {G.number_of_nodes()}")
print(f"Total edges: {G.number_of_edges()}")
print()

# Most connected compounds
print("Most connected nodes (by degree):")
degree_sorted = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:8]
for node, degree in degree_sorted:
    ntype = node_types[node]
    bar   = "█" * degree
    print(f"  {node:<30} {bar} ({degree}) [{ntype}]")

print()
print("Most prolific endophytes (compounds produced):")
for node in G.nodes():
    if node_types[node] == "Endophyte":
        out_degree = G.out_degree(node)
        if out_degree > 1:
            print(f"  {node}: {out_degree} compounds")

In [ ]:
# ── CELL 9: Download outputs ──
from google.colab import files
files.download("figures/network_graph.html")
files.download("figures/network_graph.png")
print("✓ Downloaded!")
print("  Upload BOTH to your GitHub repo under figures/")
print("  The HTML file is interactive — share it to impress recruiters!")